# DLAV Phase 3

Clean Phase 3 launcher notebook for sim-to-real generalization.
The planner uses only the front camera and `sdc_history_feature`, and predicts 60 future XY positions.


In [ ]:
from pathlib import Path

# ==================================
# Edit this cell before running
# ==================================
# Primary outputs are saved under outputs/runs/phase3/<timestamp>_<run_name>/.

# --- Colab and storage behavior ---
REPO_URL = 'https://github.com/math707/dlav-project.git'
COLAB_PROJECT_DIR = Path('/content/dlav-project')
MOUNT_DRIVE_IN_COLAB = True
COLAB_DRIVE_RUNS_ROOT = Path('/content/drive/MyDrive/dlav-project-runs')
DOWNLOAD_DATA_IF_MISSING = True
SYNC_RUN_TO_DRIVE = True
PHASE_NAME = 'phase3'

# --- Experiment identity and core training settings ---
RUN_NAME = None
MODEL_NAME = 'phase3_resnet18'
NUM_EPOCHS = 100
BATCH_SIZE = 32
TEST_BATCH_SIZE = 250
LR = 1e-3
LEARNING_RATE_NAME = 'manual'
LEARNING_RATE_OPTIONS = {LEARNING_RATE_NAME: LR}
WEIGHT_DECAY = 1e-4
REAL_TRAIN_COUNT = 500
SEED = 42

# --- Augmentation and backbone behavior ---
USE_AUGMENTATION = True
AUGMENTATION_STRENGTH = 0.5
PRETRAINED = True
BACKBONE_LEARNING_RATE = None
BACKBONE_LR_SCALE = 0.1
BACKBONE_WARMUP_EPOCHS = 2

# --- Scheduler and early stopping ---
USE_LR_SCHEDULER = True
SCHEDULER_NAME = 'plateau'
SCHEDULER_METRIC = 'val_ADE'
SCHEDULER_FACTOR = 0.5
SCHEDULER_PATIENCE = 6
SCHEDULER_MIN_LR = 1e-5
EARLY_STOPPING_PATIENCE = 25
EARLY_STOPPING_MIN_DELTA = 1e-3

# --- Inference and output behavior ---
RELOAD_BEST_CHECKPOINT_FOR_INFERENCE = True

EXPERIMENT_NAME = f'{MODEL_NAME}_realmix{REAL_TRAIN_COUNT}'
if PRETRAINED:
    EXPERIMENT_NAME += '_pretrained'
if USE_AUGMENTATION:
    EXPERIMENT_NAME += f'_aug{AUGMENTATION_STRENGTH:g}'
if USE_LR_SCHEDULER:
    EXPERIMENT_NAME += f'_{SCHEDULER_NAME}'
if WEIGHT_DECAY > 0:
    EXPERIMENT_NAME += f'_wd{WEIGHT_DECAY:g}'


In [ ]:
import os
import subprocess
import sys
from pathlib import Path


def _is_running_in_colab() -> bool:
    try:
        import google.colab  # type: ignore
        return True
    except ImportError:
        return False


def _find_project_root(start: Path) -> Path | None:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'src').is_dir() and (candidate / 'notebooks').is_dir():
            return candidate
    return None


def _bootstrap_project_root() -> tuple[Path, bool]:
    in_colab = _is_running_in_colab()
    project_root = _find_project_root(Path.cwd())

    if in_colab:
        if project_root is None:
            git_dir = COLAB_PROJECT_DIR / '.git'
            if git_dir.is_dir():
                print(f'Updating repository in {COLAB_PROJECT_DIR}...')
                subprocess.check_call(['git', '-C', str(COLAB_PROJECT_DIR), 'pull', '--ff-only'])
                project_root = COLAB_PROJECT_DIR
            elif COLAB_PROJECT_DIR.exists():
                if (COLAB_PROJECT_DIR / 'src').is_dir() and (COLAB_PROJECT_DIR / 'notebooks').is_dir():
                    print(f'Using existing project directory in {COLAB_PROJECT_DIR}...')
                    project_root = COLAB_PROJECT_DIR
                else:
                    raise FileExistsError(
                        f'{COLAB_PROJECT_DIR} exists but is not a recognized project root.'
                    )
            else:
                print(f'Cloning repository into {COLAB_PROJECT_DIR}...')
                subprocess.check_call(['git', 'clone', REPO_URL, str(COLAB_PROJECT_DIR)])
                project_root = COLAB_PROJECT_DIR
        elif project_root == COLAB_PROJECT_DIR and (COLAB_PROJECT_DIR / '.git').is_dir():
            print(f'Updating repository in {COLAB_PROJECT_DIR}...')
            subprocess.check_call(['git', '-C', str(COLAB_PROJECT_DIR), 'pull', '--ff-only'])
    elif project_root is None:
        raise FileNotFoundError(
            'Could not find the project root. Open the notebook from inside the repository.'
        )

    os.chdir(project_root)
    if str(project_root) not in sys.path:
        sys.path.insert(0, str(project_root))
    return project_root.resolve(), in_colab


PROJECT_ROOT, IN_COLAB = _bootstrap_project_root()

from src.shared.project_setup import prepare_project_context

PROJECT = prepare_project_context(
    project_root=PROJECT_ROOT,
    in_colab=IN_COLAB,
    mount_drive_in_colab=MOUNT_DRIVE_IN_COLAB,
    drive_runs_root=COLAB_DRIVE_RUNS_ROOT,
    phase_name=PHASE_NAME,
)

PROJECT_ROOT = PROJECT.project_root
DATA_DIR = PROJECT.data_dir
TRAIN_DIR = DATA_DIR / 'train'
REAL_DIR = DATA_DIR / 'val_real'
TEST_REAL_DIR = DATA_DIR / 'test_public_real'
RUNS_DIR = PROJECT.runs_dir
CHECKPOINT_DIR = PROJECT.checkpoints_dir
SUBMISSION_DIR = PROJECT.submissions_dir
LEGACY_CHECKPOINT_PATH = PROJECT.legacy_checkpoint_path
LEGACY_SUBMISSION_PATH = PROJECT.submissions_dir / 'submission_phase3.csv'

print(f'Environment: {"Google Colab" if PROJECT.in_colab else "Local"}')
print(f'Project root: {PROJECT_ROOT}')
print(f'Python executable: {sys.executable}')
print(f'Data directory: {DATA_DIR}')
print(f'Run directory root: {RUNS_DIR}')
print(f'Checkpoint directory: {CHECKPOINT_DIR}')
print(f'Submission directory: {SUBMISSION_DIR}')


In [ ]:
import os
import random

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch.utils.data import DataLoader

from src.phase3 import (
    DrivingDataset,
    PHASE3_DATASET_SPECS,
    build_model,
    build_phase3_splits,
    build_train_augmentations,
    generate_submission,
    list_test_public_real_files,
    train,
    validate,
)
from src.shared.data_utils import ensure_all_datasets, has_pkl_files
from src.shared.logger import Logger
from src.shared.run_utils import (
    build_initial_run_metrics,
    copy_artifact_to_destination,
    create_run_context,
    save_metrics,
    sync_run_to_drive,
    write_summary,
)
from src.shared.training_setup import build_optimizer, build_scheduler


def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
NUM_WORKERS = 2 if (PROJECT.in_colab or os.name != 'nt') else 0
PIN_MEMORY = DEVICE.type == 'cuda'

if DOWNLOAD_DATA_IF_MISSING:
    ensure_all_datasets(
        data_dir=DATA_DIR,
        in_colab=PROJECT.in_colab,
        dataset_specs=PHASE3_DATASET_SPECS,
        project_root=PROJECT_ROOT,
    )
else:
    expected_dirs = {
        'train': TRAIN_DIR,
        'val_real': REAL_DIR,
        'test_public_real': TEST_REAL_DIR,
    }
    missing_splits = [name for name, path in expected_dirs.items() if not has_pkl_files(path)]
    if missing_splits:
        missing_str = ', '.join(missing_splits)
        raise FileNotFoundError(
            f'Missing extracted dataset folders for: {missing_str}. '
            'Either enable DOWNLOAD_DATA_IF_MISSING or place the files manually under data/.'
        )

print(f'Device: {DEVICE}')
print(f'num_workers: {NUM_WORKERS}')
print(f'Experiment name: {EXPERIMENT_NAME}')
print(f'Model: {MODEL_NAME}')
print(f'Pretrained backbone: {PRETRAINED}')
print(f'Real train count: {REAL_TRAIN_COUNT}')
print(f'Use augmentation: {USE_AUGMENTATION} | strength={AUGMENTATION_STRENGTH}')
print(f'LR: {LR} | Weight decay: {WEIGHT_DECAY}')


In [ ]:
RUN_CONTEXT = create_run_context(
    project_root=PROJECT_ROOT,
    in_colab=PROJECT.in_colab,
    run_name=RUN_NAME,
    default_run_name=EXPERIMENT_NAME,
    drive_root=PROJECT.drive_runs_root,
    phase_name=PHASE_NAME,
)

RUN_METRICS = build_initial_run_metrics(
    RUN_CONTEXT,
    model_name=MODEL_NAME,
    device=str(DEVICE),
    batch_size=BATCH_SIZE,
    learning_rate_name=LEARNING_RATE_NAME,
    learning_rate_options=LEARNING_RATE_OPTIONS,
    learning_rate=LR,
    weight_decay=WEIGHT_DECAY,
    scheduler_enabled=USE_LR_SCHEDULER,
    scheduler_name=SCHEDULER_NAME,
    scheduler_metric=SCHEDULER_METRIC,
    num_epochs=NUM_EPOCHS,
    legacy_checkpoint_path=LEGACY_CHECKPOINT_PATH,
    legacy_submission_path=LEGACY_SUBMISSION_PATH,
)
RUN_METRICS.update(
    {
        'pretrained_backbone': PRETRAINED,
        'real_train_count': REAL_TRAIN_COUNT,
        'seed': SEED,
        'use_augmentation': USE_AUGMENTATION,
        'augmentation_strength': AUGMENTATION_STRENGTH,
        'backbone_lr_scale': BACKBONE_LR_SCALE,
        'backbone_warmup_epochs': BACKBONE_WARMUP_EPOCHS,
    }
)

save_metrics(RUN_CONTEXT, RUN_METRICS)
write_summary(RUN_CONTEXT, RUN_METRICS)

print(f'Run name: {RUN_CONTEXT.run_name}')
print(f'Run directory: {RUN_CONTEXT.run_dir}')
print(f'Best checkpoint path: {RUN_CONTEXT.checkpoint_path}')
print(f'Submission path: {RUN_CONTEXT.submission_path}')


In [ ]:
SPLITS = build_phase3_splits(
    TRAIN_DIR,
    REAL_DIR,
    real_train_count=REAL_TRAIN_COUNT,
    seed=SEED,
)
if not SPLITS['real_val']:
    raise ValueError('real_val split is empty. Reduce REAL_TRAIN_COUNT to keep validation samples.')

train_transform = build_train_augmentations(AUGMENTATION_STRENGTH) if USE_AUGMENTATION else None

train_dataset = DrivingDataset(
    SPLITS['mixed_train'],
    image_transform=train_transform,
    future_xy_only=True,
)
val_dataset = DrivingDataset(SPLITS['real_val'], future_xy_only=True)
test_files = list_test_public_real_files(TEST_REAL_DIR)
test_dataset = DrivingDataset(test_files, test=True)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=TEST_BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)

sample = val_dataset[0]
RUN_METRICS.update(
    {
        'synthetic_train_count': len(SPLITS['synthetic_train']),
        'real_train_count_actual': len(SPLITS['real_train']),
        'real_val_count': len(SPLITS['real_val']),
        'mixed_train_count': len(SPLITS['mixed_train']),
        'test_public_real_count': len(test_dataset),
    }
)
save_metrics(RUN_CONTEXT, RUN_METRICS)
write_summary(RUN_CONTEXT, RUN_METRICS)

print(f"Synthetic train samples: {len(SPLITS['synthetic_train'])}")
print(f"Real train samples: {len(SPLITS['real_train'])}")
print(f"Real val samples: {len(SPLITS['real_val'])}")
print(f"Mixed train samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")
print(f"camera: {tuple(sample['camera'].shape)} | history: {tuple(sample['history'].shape)} | future: {tuple(sample['future'].shape)}")


In [ ]:
preview_count = min(4, len(val_dataset))
fig, axes = plt.subplots(2, preview_count, figsize=(4 * preview_count, 8))
if preview_count == 1:
    axes = np.array(axes).reshape(2, 1)

for index in range(preview_count):
    sample = val_dataset[index]
    camera = sample['camera'].permute(1, 2, 0).cpu().numpy().clip(0.0, 1.0)
    history = sample['history'].cpu().numpy()
    future = sample['future'].cpu().numpy()

    axes[0, index].imshow(camera)
    axes[0, index].set_title(f'Real sample {index}')
    axes[0, index].axis('off')

    axes[1, index].plot(history[:, 0], history[:, 1], 'o-', color='gold', label='Past')
    axes[1, index].plot(future[:, 0], future[:, 1], 'o-', color='green', label='Future')
    axes[1, index].axis('equal')
    axes[1, index].set_title('Trajectory')
    axes[1, index].legend()

plt.tight_layout()
plt.show()


In [ ]:
model = build_model(
    MODEL_NAME,
    pretrained_backbone=PRETRAINED,
)

optimizer = build_optimizer(
    model,
    learning_rate=LR,
    weight_decay=WEIGHT_DECAY,
    backbone_learning_rate=BACKBONE_LEARNING_RATE,
    backbone_lr_scale=BACKBONE_LR_SCALE,
)
scheduler = build_scheduler(
    optimizer,
    enabled=USE_LR_SCHEDULER,
    name=SCHEDULER_NAME,
    factor=SCHEDULER_FACTOR,
    patience=SCHEDULER_PATIENCE,
    min_lr=SCHEDULER_MIN_LR,
)
logger = Logger(log_path=RUN_CONTEXT.log_path)
LAST_CHECKPOINT_PATH = RUN_CONTEXT.run_dir / 'model_last.pth'

TRAINING_SUMMARY = train(
    model,
    train_loader,
    val_loader,
    optimizer,
    logger,
    num_epochs=NUM_EPOCHS,
    scheduler=scheduler,
    scheduler_metric=SCHEDULER_METRIC,
    best_checkpoint_path=RUN_CONTEXT.checkpoint_path,
    last_checkpoint_path=LAST_CHECKPOINT_PATH,
    early_stopping_patience=EARLY_STOPPING_PATIENCE,
    early_stopping_min_delta=EARLY_STOPPING_MIN_DELTA,
    backbone_warmup_epochs=BACKBONE_WARMUP_EPOCHS,
)

final_metrics = TRAINING_SUMMARY.get('final', {})
best_metrics = TRAINING_SUMMARY.get('best', {})
RUN_METRICS.update(
    {
        'epochs_completed': TRAINING_SUMMARY.get('epochs_completed'),
        'train_loss_final': final_metrics.get('train_loss'),
        'val_loss_final': final_metrics.get('val_loss'),
        'val_ADE_final': final_metrics.get('val_ADE'),
        'val_FDE_final': final_metrics.get('val_FDE'),
        'best_val_ADE': best_metrics.get('val_ADE'),
        'best_val_ADE_epoch': best_metrics.get('epoch'),
        'best_val_FDE_at_best_ADE': best_metrics.get('val_FDE'),
        'best_val_loss_at_best_ADE': best_metrics.get('val_loss'),
        'best_checkpoint_path': best_metrics.get('checkpoint_path'),
        'checkpoint_path': best_metrics.get('checkpoint_path'),
        'last_checkpoint_path': str(LAST_CHECKPOINT_PATH),
        'epoch_history': TRAINING_SUMMARY.get('history', []),
        'final_learning_rate': TRAINING_SUMMARY.get('final_learning_rate'),
    }
)
save_metrics(RUN_CONTEXT, RUN_METRICS)
write_summary(RUN_CONTEXT, RUN_METRICS)

if best_metrics:
    print(f"Best val ADE: {best_metrics['val_ADE']:.4f} at epoch {best_metrics['epoch']}")
    print(f"Best checkpoint: {best_metrics.get('checkpoint_path')}")
print(f'Run metrics updated: {RUN_CONTEXT.metrics_path}')
print(f'Run log: {RUN_CONTEXT.log_path}')


In [ ]:
copy_artifact_to_destination(RUN_CONTEXT.checkpoint_path, LEGACY_CHECKPOINT_PATH)

if RELOAD_BEST_CHECKPOINT_FOR_INFERENCE:
    best_state_dict = torch.load(RUN_CONTEXT.checkpoint_path, map_location=DEVICE)
    model.load_state_dict(best_state_dict)
    model = model.to(DEVICE)
    RUN_METRICS['inference_checkpoint_path'] = str(RUN_CONTEXT.checkpoint_path)
else:
    RUN_METRICS['inference_checkpoint_path'] = str(LAST_CHECKPOINT_PATH)

VALIDATION_METRICS = validate(model, val_loader, device=DEVICE)
RUN_METRICS.update(
    {
        'reloaded_val_ADE': VALIDATION_METRICS.get('val_ADE'),
        'reloaded_val_FDE': VALIDATION_METRICS.get('val_FDE'),
        'reloaded_val_loss': VALIDATION_METRICS.get('val_loss'),
    }
)
save_metrics(RUN_CONTEXT, RUN_METRICS)
write_summary(RUN_CONTEXT, RUN_METRICS)

print(f'Best model saved to {RUN_CONTEXT.checkpoint_path}')
print(f'Last-epoch model saved to {LAST_CHECKPOINT_PATH}')
print(f'Historical checkpoint copy saved to {LEGACY_CHECKPOINT_PATH}')
print(VALIDATION_METRICS)


In [ ]:
submission = generate_submission(
    model=model,
    output_path=RUN_CONTEXT.submission_path,
    device=DEVICE,
    data_loader=test_loader,
    legacy_output_path=LEGACY_SUBMISSION_PATH,
    copy_fn=copy_artifact_to_destination,
    expected_num_samples=1000,
)

RUN_METRICS['submission_path'] = str(RUN_CONTEXT.submission_path)
RUN_METRICS['submission_shape'] = list(submission.shape)
save_metrics(RUN_CONTEXT, RUN_METRICS)
write_summary(RUN_CONTEXT, RUN_METRICS)

drive_backup_dir = sync_run_to_drive(RUN_CONTEXT) if SYNC_RUN_TO_DRIVE else None
if drive_backup_dir is not None:
    RUN_METRICS['drive_backup_enabled'] = True
    RUN_METRICS['drive_backup_path'] = str(drive_backup_dir)
    save_metrics(RUN_CONTEXT, RUN_METRICS)
    write_summary(RUN_CONTEXT, RUN_METRICS)

print(f'Submission saved to {RUN_CONTEXT.submission_path}')
print(f'Historical submission copy saved to {LEGACY_SUBMISSION_PATH}')
if drive_backup_dir is not None:
    print(f'Run backup synchronized to {drive_backup_dir}')
print(f'Submission shape: {submission.shape}')
